# 03 — Inequality Analysis

For each of the 92 LMICs, this notebook:
1. Clips and aligns population, RWI, and productivity loss rasters to the country
2. Calculates the concentration index (CI) and quantile ratio
3. Generates concentration curve data
4. Saves all results to `results/inequality_results.csv` and `results/concentration_curves.csv`

**Prerequisite:** run notebook 02 to generate the productivity loss raster.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from scripts.inequality import calculate_CI, calculate_concentration_curve, calculate_quantile_ratio, prepare_arrays
from scripts.raster_utils import get_country_geometry, align_rasters

with open(ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

print('Ready.')

## Configuration — select epoch

In [ ]:
# Match the epoch you processed in notebook 02
EPOCH_START = config['wbgt']['start_year']
EPOCH_END   = config['wbgt']['end_year']

POP_PATH        = str(DATA / config['data']['population'])
RWI_PATH        = str(DATA / config['data']['rwi'])
RISK_PATH       = str(DATA / 'processed' / f'productivity_loss_{EPOCH_START}_{EPOCH_END}.tif')
BOUNDARIES_PATH = str(DATA / config['data']['boundaries'])

ISO_CODES = config['iso_codes']

print(f'Epoch:    {EPOCH_START}–{EPOCH_END}')
print(f'Risk map: {RISK_PATH}')
print(f'Countries: {len(ISO_CODES)}')

## Run analysis for all countries

In [ ]:
ci_results = []
curve_records = []

for iso in tqdm(ISO_CODES, desc='Countries'):
    try:
        # Get country boundary
        geom = get_country_geometry(BOUNDARIES_PATH, iso)

        # Clip and align all rasters to the population grid
        pop, rwi, risk = align_rasters(POP_PATH, RWI_PATH, RISK_PATH, geom)

        # Clean arrays (remove NaNs, zero-pop cells)
        arrays = prepare_arrays(pop, rwi, risk)
        if arrays is None:
            print(f'  {iso}: insufficient valid data — skipping')
            continue
        pop_f, rwi_f, risk_f = arrays

        # Inequality metrics
        ci = calculate_CI(pop_f, rwi_f, risk_f)
        qr = calculate_quantile_ratio(pop_f, rwi_f, risk_f, quantile=0.2)

        # Population-weighted mean productivity loss
        mean_loss = float(np.average(risk_f, weights=pop_f))
        total_pop = float(pop_f.sum())
        pop_coverage = float(pop_f.sum() / np.nansum(pop))  # share of pop with RWI data

        ci_results.append({
            'iso3': iso,
            'epoch_start': EPOCH_START,
            'epoch_end': EPOCH_END,
            'CI': ci,
            'QR_Q1_Q5': qr,
            'mean_productivity_loss': mean_loss,
            'total_population': total_pop,
            'rwi_coverage': pop_coverage,
            'n_cells': len(pop_f),
        })

        # Concentration curve
        cum_pop, cum_risk = calculate_concentration_curve(pop_f, rwi_f, risk_f)
        for cp, cr in zip(cum_pop, cum_risk):
            curve_records.append({'iso3': iso, 'cum_pop_share': cp, 'cum_risk_share': cr})

    except Exception as e:
        print(f'  {iso}: ERROR — {e}')

ci_df = pd.DataFrame(ci_results)
curves_df = pd.DataFrame(curve_records)

print(f'\nCompleted: {len(ci_df)} / {len(ISO_CODES)} countries')
ci_df.sort_values('CI').head(10)

## Save results

In [ ]:
ci_out = RESULTS / f'inequality_results_{EPOCH_START}_{EPOCH_END}.csv'
curves_out = RESULTS / f'concentration_curves_{EPOCH_START}_{EPOCH_END}.csv'

ci_df.to_csv(ci_out, index=False)
curves_df.to_csv(curves_out, index=False)

print(f'Saved:\n  {ci_out}\n  {curves_out}')

## Quick summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CI distribution
ax = axes[0]
ax.hist(ci_df['CI'].dropna(), bins=20, color='steelblue', edgecolor='white')
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Concentration Index')
ax.set_ylabel('Number of countries')
ax.set_title('Distribution of CI across 92 LMICs')

# Mean productivity loss vs CI
ax = axes[1]
sc = ax.scatter(ci_df['mean_productivity_loss'], ci_df['CI'],
                c=ci_df['rwi_coverage'], cmap='RdYlGn', edgecolors='grey', linewidth=0.5)
plt.colorbar(sc, ax=ax, label='RWI coverage')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Mean annual productivity loss')
ax.set_ylabel('Concentration Index')
ax.set_title('Exposure vs Inequality')

plt.tight_layout()
plt.show()